# Discover recurrent molecular events across donors

This notebook searches a rooted Gravlax collection without supplying event coordinates. Candidate discovery uses collection metadata; exact support is reduced from routed source-archive molecule records. The demo manifest supplies the donor design, cell groups, thresholds, and immutable assets. Its final figure is a deterministic SVG built from the typed result tables using only the Python standard library and the IPython display API supplied by Colab.

In [ ]:
#@title Immutable demo manifest (required)
MANIFEST_URL = "https://github.com/COMBINE-lab/gravlax/releases/download/demo-data-v1/demo-manifest.json" #@param {type:"string"}
MANIFEST_SHA256 = "82c34aad442d478f1cb1243a6ccfe8ad9f937b81d9e1f946a8eb2cfc498214fd" #@param {type:"string"}
if not MANIFEST_URL or not MANIFEST_SHA256:
    raise RuntimeError("Demo capsule not published/configured: set the immutable manifest URL and its SHA-256. No fallback URL is used.")

In [ ]:
import hashlib, json, re, subprocess, sys, tarfile, urllib.parse, urllib.request, zipfile
from pathlib import Path
WORK = Path('/content/gravlax-demo'); WORK.mkdir(parents=True, exist_ok=True)
HEX64 = re.compile(r'^[0-9a-f]{64}$')
def download_verified(url, sha256, destination):
    parsed_url = urllib.parse.urlsplit(url) if isinstance(url, str) else None
    if parsed_url is None or parsed_url.scheme != 'https' or not parsed_url.netloc: raise ValueError(f'asset URL must be HTTPS, got {url!r}')
    if 'latest' in (segment.lower() for segment in parsed_url.path.split('/')): raise ValueError(f'asset URL must be immutable; /latest/ is not allowed: {url!r}')
    if not isinstance(sha256, str) or not HEX64.fullmatch(sha256): raise ValueError('asset SHA-256 must be 64 lowercase hexadecimal characters')
    destination = Path(destination); temporary = destination.with_suffix(destination.suffix + '.part'); digest = hashlib.sha256()
    with urllib.request.urlopen(url) as source, temporary.open('wb') as sink:
        while block := source.read(1 << 20): digest.update(block); sink.write(block)
    if digest.hexdigest() != sha256:
        temporary.unlink(missing_ok=True); raise RuntimeError(f'SHA-256 mismatch for {url}')
    temporary.replace(destination); return destination
manifest_path = download_verified(MANIFEST_URL, MANIFEST_SHA256, WORK / 'manifest.json')
manifest = json.loads(manifest_path.read_text())
if manifest.get('schema') != 'gravlax.demo-capsule.v1': raise RuntimeError('unsupported or missing demo manifest schema')
for section in ('software', 'resources', 'stories'):
    if not isinstance(manifest.get(section), dict): raise RuntimeError(f'manifest {section} must be an object')
required_software_fields = {'version', 'aie', 'python_wheel'}
if required_software_fields.difference(manifest['software']): raise RuntimeError(f'manifest software lacks {sorted(required_software_fields.difference(manifest["software"]))}')
RESERVED_ASSET_FILENAMES = {'.', '..', 'manifest.json', 'event-discovery.aicollection', 'junction-drilldown.aicollection'}
asset_filenames = {}
for asset_name, asset_spec in [('software.aie', manifest['software'].get('aie')), ('software.python_wheel', manifest['software'].get('python_wheel')), *[(f'resources.{name}', spec) for name, spec in manifest['resources'].items()]]:
    if not isinstance(asset_spec, dict): raise RuntimeError(f'asset declaration {asset_name} must be an object')
    filename = asset_spec.get('filename')
    if not isinstance(filename, str) or not filename or Path(filename).name != filename or filename in RESERVED_ASSET_FILENAMES: raise ValueError(f'asset declaration {asset_name} has an invalid or reserved filename: {filename!r}')
    if filename in asset_filenames: raise ValueError(f'duplicate asset filename {filename!r}: {asset_filenames[filename]} and {asset_name}')
    asset_filenames[filename] = asset_name
def fetch(spec):
    if not isinstance(spec, dict) or any(not spec.get(key) for key in ('url','sha256','filename')): raise RuntimeError(f'incomplete published asset declaration: {spec!r}')
    if Path(spec['filename']).name != spec['filename']: raise ValueError('asset filename must be a basename')
    return download_verified(spec['url'], spec['sha256'], WORK / spec['filename'])
def install_tools():
    wheel = fetch(manifest['software']['python_wheel']); subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', '--no-deps', str(wheel)], check=True)
    spec = manifest['software']['aie']; bundle = fetch(spec); member = spec.get('member')
    if member:
        if zipfile.is_zipfile(bundle):
            with zipfile.ZipFile(bundle) as archive: payload = archive.read(member)
        else:
            with tarfile.open(bundle, 'r:*') as archive:
                item = archive.getmember(member)
                if not item.isfile(): raise RuntimeError('configured aie archive member is not a file')
                payload = archive.extractfile(item).read()
        binary = WORK / 'aie'; binary.write_bytes(payload)
    else: binary = bundle
    binary.chmod(0o755); return binary
AIE = install_tools()
from gravlax import Client, __version__ as PYTHON_VERSION
EXPECTED_VERSION = manifest['software'].get('version')
if not isinstance(EXPECTED_VERSION, str) or not EXPECTED_VERSION: raise RuntimeError('manifest software.version must be nonempty')
CLI_VERSION = subprocess.run([str(AIE), '--version'], check=True, capture_output=True, text=True).stdout.strip()
if CLI_VERSION != f'aie {EXPECTED_VERSION}': raise RuntimeError(f'CLI version mismatch: {CLI_VERSION!r} != aie {EXPECTED_VERSION}')
if PYTHON_VERSION != EXPECTED_VERSION: raise RuntimeError(f'Python version mismatch: {PYTHON_VERSION!r} != {EXPECTED_VERSION}')
client = Client(binary=AIE); print(CLI_VERSION)

In [ ]:
story = manifest['stories'].get('event_discovery'); resources = manifest['resources']
required_story_fields = {'archives', 'collection_groups', 'design', 'annotation', 'assembly', 'annotation_label', 'comparison_annotation', 'comparison_annotation_label', 'expected_entity_id', 'expected_min_exact_umi_classes', 'expected_min_exact_donors', 'expected_gap_primary_class', 'expected_annotation_incompatible', 'expected_rank', 'expected_comparison_compatible_transcripts'}
if not isinstance(story, dict) or required_story_fields.difference(story): raise RuntimeError(f'event_discovery story lacks {sorted(required_story_fields.difference(story or {}))}')
if story.get('annotation') and {'assembly', 'annotation_label'}.difference(story): raise RuntimeError('event_discovery annotation requires assembly and annotation_label')
if not isinstance(story['archives'], dict) or not story['archives']: raise RuntimeError('event_discovery.archives must be a nonempty sample-to-resource object')
def story_resource(name):
    spec = resources.get(name)
    if not isinstance(name, str) or not isinstance(spec, dict): raise RuntimeError(f'story resource is not declared: {name!r}')
    return spec
COLLECTION_ROOT_PATTERN = re.compile(r'^aicollection-directory-root-v1:[0-9a-f]{64}$')
def prebuilt_collection(story):
    """Return the capsule's published collection and location manifest, or (None, None).

    A v1 capsule declares no collection, so the notebook rebuilds one locally. When the
    manifest declares one, every committed source identity is resolved through the
    hash-pinned location manifest whose relative paths are read beside its own download.
    """
    declared = manifest.get('collection')
    if declared is None: return None, None
    if not isinstance(declared, dict) or set(declared) != {'archives', 'shape_routes', 'allow_unstamped', 'collection_root', 'asset', 'locations'}: raise RuntimeError('manifest collection has missing or unknown fields')
    if declared['archives'] != story['archives']: raise RuntimeError('the published collection does not commit exactly this story\'s archives')
    if declared['shape_routes'] is not story.get('shape_routes', True) or declared['allow_unstamped'] is not story.get('allow_unstamped', False): raise RuntimeError('the published collection was not built with this story\'s collection options')
    if not isinstance(declared['collection_root'], str) or not COLLECTION_ROOT_PATTERN.fullmatch(declared['collection_root']): raise RuntimeError('manifest collection lacks a rooted content identity')
    for label, suffix in (('asset', '.aicollection'), ('locations', '.json')):
        spec = declared[label]
        filename = spec.get('filename') if isinstance(spec, dict) else None
        if not isinstance(filename, str) or Path(filename).name != filename or not filename.endswith(suffix) or filename in RESERVED_ASSET_FILENAMES or filename in asset_filenames: raise RuntimeError(f'manifest collection.{label} has an invalid, reserved, or duplicate filename: {filename!r}')
    if declared['asset']['filename'] == declared['locations']['filename']: raise RuntimeError('the collection and its location manifest must be distinct files')
    collection_path = fetch(declared['asset']); locations_path = fetch(declared['locations'])
    document = json.loads(locations_path.read_text())
    if not isinstance(document, dict) or set(document) != {'schema_version', 'locations'} or document['schema_version'] != 1: raise RuntimeError('unsupported location manifest schema')
    expected = {}
    for resource in story['archives'].values():
        spec = story_resource(resource)
        if expected.setdefault(spec['archive_root'], spec['filename']) != spec['filename']: raise RuntimeError('two declared collection sources share one archive root')
    observed = {}
    for entry in document.get('locations') or ():
        if not isinstance(entry, dict) or set(entry) != {'identity', 'path'}: raise RuntimeError('location entry has missing or unknown fields')
        if entry['identity'] in observed: raise RuntimeError('the location manifest repeats an identity')
        if not isinstance(entry['path'], str) or Path(entry['path']).name != entry['path']: raise RuntimeError('location paths must be capsule basenames resolved beside the manifest')
        observed[entry['identity']] = entry['path']
    if observed != expected: raise RuntimeError('the location manifest does not resolve exactly the downloaded story archives')
    for sample, resource in story['archives'].items():
        if (locations_path.parent / observed[story_resource(resource)['archive_root']]).resolve() != Path(archives[sample]).resolve(): raise RuntimeError(f'the location manifest does not point at the verified download for {sample}')
    inspected = client.result_raw(['collection', 'inspect', collection_path, f'--locations={locations_path}', '--verify-routes'])
    layers = inspected.get('layers')
    if not isinstance(layers, list) or len(layers) != 1: raise RuntimeError('the published collection must be a single rooted layer')
    if f"aicollection-directory-root-v1:{layers[0].get('root_digest')}" != declared['collection_root']: raise RuntimeError('the published collection root does not match the manifest')
    guard = inspected['guard']
    if guard.get('content_identity_verified') is not True or (declared['shape_routes'] and guard.get('shape_route_reconstruction_verified') is not True): raise RuntimeError('the published collection did not authenticate its committed sources and shape routes')
    resolved = {entry['id']: f"{entry['native_identity']['scheme']}:{entry['native_identity']['blake3']}" for entry in inspected['archives']}
    if resolved != {sample: story_resource(resource)['archive_root'] for sample, resource in story['archives'].items()}: raise RuntimeError('the published collection does not commit the verified archive roots')
    if declared['shape_routes'] and inspected['index'].get('shape_route_archives') != len(resolved): raise RuntimeError('the published collection lacks a shape route for every committed archive')
    print(f"resolved the published collection through {locations_path.name}: {declared['collection_root']}")
    return collection_path, locations_path
archives = {sample: fetch(story_resource(resource)) for sample, resource in story['archives'].items()}
for sample, path in archives.items():
    expected = story_resource(story['archives'][sample]).get('archive_root')
    if not expected: raise RuntimeError(f'archive {sample} lacks a rooted identity in the manifest')
    identity = client.result_raw(['inspect-archive', path, '--json'])['native_identity']
    observed = f"{identity['scheme']}:{identity['blake3']}"
    if observed != expected: raise RuntimeError(f'archive root mismatch for {sample}: {observed}')
collection, locations = prebuilt_collection(story)
if collection is None:
    collection = WORK / 'event-discovery.aicollection'; collection.unlink(missing_ok=True)
    build = ['collection', 'build']
    for sample, path in sorted(archives.items()): build.append(f'--sample={sample}={path}')
    if story.get('shape_routes', True): build.append('--shape-routes')
    if story.get('allow_unstamped', False): build.append('--allow-unstamped')
    build.extend([f'--out={collection}'])
    client.run(build)
groups = fetch(story_resource(story['collection_groups']))
design = fetch(story_resource(story['design']))
annotation = fetch(story_resource(story['annotation'])) if story.get('annotation') else None
comparison_annotation = fetch(story_resource(story['comparison_annotation']))
result = client.collection_find_events(
    collection, locations=locations, kinds=tuple(story.get('kinds', ())), design=design, groups=groups,
    require_groups=tuple(story.get('require_groups', ())),
    min_group_umi_classes=story.get('min_group_umi_classes', 1), min_donors=story.get('min_donors', 1),
    min_samples=story.get('min_samples', 1), min_umi_classes=story.get('min_umi_classes', 1),
    min_side_umi_classes=story.get('min_side_umi_classes', 1), min_support=story.get('min_support', 2),
    terminal_cluster_bp=story.get('terminal_cluster_bp', 25),
    max_terminal_events=story.get('max_terminal_events', 10000000),
    annotation=annotation, assembly=story.get('assembly'),
    annotation_label=story.get('annotation_label'), annotation_digest=story.get('annotation_digest'),
    novel_only=story.get('novel_only', False),
    solo_strand=story.get('solo_strand', 'forward'),
    max_candidates=story.get('max_candidates', 100000),
    max_candidates_considered=story.get('max_candidates_considered', 1000000),
    max_routed_entries=story.get('max_routed_entries', 10000000),
    max_exact_match_attempts=story.get('max_exact_match_attempts', 25000000),
    max_annotation_comparisons=story.get('max_annotation_comparisons', 10000000),
)
print(json.dumps(result.summary.as_dict(), indent=2))
print('tables:', result.table_names)
required_tables = {'capabilities', 'entities', 'components', 'counts', 'terminal_anchors', 'terminal_counts'}
missing_tables = required_tables.difference(result.table_names)
if missing_tables: raise RuntimeError(f'find-events result is missing tables: {sorted(missing_tables)}')
entities = result.table('entities').records()
locked_rows = [row for row in entities if row.get('entity_id') == story['expected_entity_id']]
if len(locked_rows) != 1: raise RuntimeError(f"expected exactly one locked event {story['expected_entity_id']}")
locked = locked_rows[0]; locked_rank = entities.index(locked) + 1
if locked_rank != story['expected_rank']: raise RuntimeError(f"locked event rank {locked_rank} != {story['expected_rank']}")
if locked['exact_umi_classes'] < story['expected_min_exact_umi_classes']: raise RuntimeError('locked event exact UMI support fell below its minimum')
if locked['exact_donors'] < story['expected_min_exact_donors']: raise RuntimeError('locked event donor support fell below its minimum')
if locked.get('gap_primary_class') != story['expected_gap_primary_class']: raise RuntimeError('locked event gap classification changed')
if locked.get('annotation_incompatible') is not story['expected_annotation_incompatible']: raise RuntimeError('locked event annotation compatibility changed')
print(f"verified locked event at rank {locked_rank}: {locked['entity_id']}")
comparison_result = client.collection_find_events(
    collection, locations=locations, kinds=tuple(story.get('kinds', ())), design=design, groups=groups,
    require_groups=tuple(story.get('require_groups', ())),
    min_group_umi_classes=story.get('min_group_umi_classes', 1), min_donors=story.get('min_donors', 1),
    min_samples=story.get('min_samples', 1), min_umi_classes=story.get('min_umi_classes', 1),
    min_side_umi_classes=story.get('min_side_umi_classes', 1), min_support=story.get('min_support', 2),
    terminal_cluster_bp=story.get('terminal_cluster_bp', 25),
    max_terminal_events=story.get('max_terminal_events', 10000000),
    annotation=comparison_annotation, assembly=story['assembly'],
    annotation_label=story['comparison_annotation_label'], novel_only=False,
    solo_strand=story.get('solo_strand', 'forward'),
    max_candidates=story.get('max_candidates', 100000),
    max_candidates_considered=story.get('max_candidates_considered', 1000000),
    max_routed_entries=story.get('max_routed_entries', 10000000),
    max_exact_match_attempts=story.get('max_exact_match_attempts', 25000000),
    max_annotation_comparisons=story.get('max_annotation_comparisons', 10000000),
)
comparison_rows = [row for row in comparison_result.table('entities').records() if row.get('entity_id') == story['expected_entity_id']]
if len(comparison_rows) != 1: raise RuntimeError('comparison annotation did not retain exactly one locked event')
comparison_locked = comparison_rows[0]
if comparison_locked.get('annotation_incompatible') is not False: raise RuntimeError('locked event remains incompatible with the comparison annotation')
if comparison_locked.get('compatible_transcripts') != story['expected_comparison_compatible_transcripts']: raise RuntimeError('locked event comparison-compatible transcript count changed')
print(f"verified {comparison_locked['compatible_transcripts']} compatible transcripts in {story['comparison_annotation_label']}")
for entity in entities[:20]: print(entity)

In [ ]:
# Deterministic SVG generated only from the typed entities and counts tables.
from html import escape
from IPython.display import SVG, display

def require_table_columns(table, expected, label):
    missing = set(expected).difference(table.columns)
    if missing: raise RuntimeError(f'{label} table is missing columns: {sorted(missing)}')

def support_bar_svg(items, title, value_label, empty_message):
    width, label_width, plot_width, row_height = 940, 330, 500, 28
    items = items[:20]; height = 92 + row_height * max(1, len(items))
    peak = max((value for _, value in items), default=1) or 1
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" role="img" aria-label="{escape(title)}">',
             '<style>text{font-family:system-ui,sans-serif;font-size:12px}.title{font-size:16px;font-weight:600}.axis{fill:#5b6470}</style>',
             f'<text class="title" x="8" y="22">{escape(title)}</text>',
             f'<text class="axis" x="{label_width}" y="43">{escape(value_label)}</text>']
    if not items: parts.append(f'<text x="8" y="72">{escape(empty_message)}</text>')
    for index, (label, value) in enumerate(items):
        y = 58 + index * row_height; span = value * plot_width / peak
        parts.extend([f'<text x="{label_width - 8}" y="{y + 14}" text-anchor="end">{escape(str(label)[:48])}</text>',
                      f'<rect x="{label_width}" y="{y}" width="{max(span, 1)}" height="18" rx="2" fill="#3366cc"/>',
                      f'<text x="{label_width + span + 5}" y="{y + 14}">{value}</text>'])
    parts.append('</svg>'); return ''.join(parts)

entity_table, count_table = result.table('entities'), result.table('counts')
require_table_columns(entity_table, {'entity_id', 'kind', 'exact_umi_classes', 'exact_samples', 'exact_donors'}, 'entities')
require_table_columns(count_table, {'entity_id', 'donor', 'group', 'informative_umi_classes'}, 'counts')
ranked_entities = sorted(entity_table.records(), key=lambda row: (-row['exact_donors'], -row['exact_samples'], -row['exact_umi_classes'], row['entity_id']))
focus = ranked_entities[0] if ranked_entities else None
support = {}
if focus:
    for row in count_table.records():
        if row['entity_id'] == focus['entity_id']:
            label = f"{row['donor']} / {row['group']}"; support[label] = support.get(label, 0) + row['informative_umi_classes']
support_rows = sorted(support.items(), key=lambda item: (-item[1], item[0]))
title = f"Recurrent support: {focus['kind']} · {focus['entity_id']}" if focus else 'Recurrent event support'
display(SVG(data=support_bar_svg(support_rows, title, 'exact raw-UMI-value classes by donor / group', 'No event passed the manifest thresholds.')))